# Module 1: Build the Anchor Transformer + Training Skeleton

In this module, you'll build a small GPT-style transformer language model from scratch and wrap it in a **production-grade training pipeline** with:

- Reproducible setup (seed everything)
- Proper data loading with tokenization
- A clean training loop with evaluation
- Checkpointing and structured logging

By the end, you'll have a working language model that generates text - and more importantly, a training setup you can reuse for any project.

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/arj7192/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete - GPU:", os.environ.get("COLAB_GPU", "not detected"))

## 1.1 Environment Setup

Two things every production script does first: **seed everything** and **auto-detect hardware**.

`set_seed(42)` seeds four separate RNG sources: Python's `random`, NumPy, PyTorch CPU, and PyTorch CUDA. It also sets `cudnn.deterministic = True` and `cudnn.benchmark = False`. If you skip any of these, your runs won't be reproducible - dropout, data shuffling, and weight initialization all use different RNG sources.

`get_device()` checks CUDA, then Apple MPS, then falls back to CPU. Write it once, and the same code runs on any hardware without `if` statements scattered through your training loop.

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f"PyTorch {torch.__version__}")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1.2 Understanding the Model Architecture

We're building a **decoder-only transformer** - the same architecture family as GPT-2/3/4, LLaMA, and Mistral. "Decoder-only" means we use only the decoder half of the original Transformer (Vaswani et al., 2017). For autoregressive language modeling - predicting the next token given all previous tokens - the encoder isn't needed.

**Key design choices (and why they matter for production):**

- **Pre-norm** (`norm_first=True`) - LayerNorm is applied *before* attention/FFN instead of after. This lets gradients flow more cleanly through the residual path, making training more stable. You can use higher learning rates and need less warmup. GPT-3, LLaMA, and PaLM all use pre-norm.

- **Weight tying** - The embedding matrix (token ID to vector) and the output projection (hidden state to vocab logits) share the same weight matrix. This cuts ~2M parameters from our model and acts as a regularizer - the space for "understanding a token" and "predicting a token" is forced to be consistent. Used by GPT-2 and BERT.

- **Causal masking** - Each token can only attend to itself and previous tokens, never future ones. This is what makes the model autoregressive. The mask is cached so we don't recreate it every forward pass.

- **Embedding scaling** (`* sqrt(d_model)`) - Without this, the positional encoding signal (values in [-1, 1]) would dominate the token embeddings (values ~0.06 after Xavier init). Scaling by sqrt(256) = 16 brings them to the same magnitude.

Our model has ~5M parameters: small enough to train in minutes on a free Colab GPU, large enough to learn real English patterns. The architecture is identical to GPT-2 - only the dimensions are smaller.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al., 2017).
    
    Transformers treat input as a set, not a sequence - they have no inherent
    notion of order. This adds a unique frequency-based "fingerprint" to each
    position so the model can tell "the cat sat" from "sat the cat".
    
    Frequencies range from wavelength 2*pi (dim 0) to ~20000*pi (dim d_model-1),
    giving each position a unique combination of sin/cos values.
    
    Alternative: learned positional embeddings (GPT-2) or RoPE (LLaMA).
    Sinusoidal is simpler and can theoretically extrapolate to longer sequences.
    """

    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # register_buffer: saved with state_dict but not a learnable parameter
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class TransformerLM(nn.Module):
    """
    Decoder-only transformer language model.
    
    This is the anchor model we'll optimize throughout the workshop.
    """

    def __init__(self, vocab_size, d_model=256, n_heads=4, d_ff=512,
                 n_layers=4, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_seq_len, dropout=dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: embedding (token -> vector) and output projection (vector -> logits)
        # share the same matrix. Saves ~2M params and improves generalization.
        self.output_proj.weight = self.token_emb.weight

        self._init_weights()
        self._causal_mask_cache = {}  # Avoid recreating the same mask every forward pass

    def _init_weights(self):
        # Xavier uniform: scales init variance by 1/fan_in, preventing signal
        # from exploding or vanishing through layers. Only for matrices (dim > 1).
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _get_causal_mask(self, seq_len, device):
        key = (seq_len, device)
        if key not in self._causal_mask_cache:
            self._causal_mask_cache[key] = nn.Transformer.generate_square_subsequent_mask(
                seq_len, device=device
            )
        return self._causal_mask_cache[key]

    def forward(self, input_ids, targets=None):
        seq_len = input_ids.size(1)
        causal_mask = self._get_causal_mask(seq_len, input_ids.device)

        # Scale embeddings by sqrt(d_model) so they're on the same scale
        # as the positional encoding (otherwise pos_enc dominates at init)
        x = self.token_emb(input_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        # Decoder-only trick: we feed a dummy "memory" tensor because PyTorch's
        # TransformerDecoder expects encoder output. All real work happens in
        # self-attention via tgt_mask (the causal mask).
        memory = torch.zeros(input_ids.size(0), 1, self.d_model, device=input_ids.device)
        x = self.transformer(x, memory, tgt_mask=causal_mask)
        logits = self.output_proj(x)

        result = {'logits': logits}
        if targets is not None:
            # Flatten (batch, seq, vocab) -> (batch*seq, vocab) for cross_entropy.
            # ignore_index=-100 lets us mask out padding tokens if needed.
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100
            )
            result['loss'] = loss
        return result

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=0.8, top_k=40):
        """
        Autoregressive generation: predict one token at a time, append, repeat.
        
        - temperature: <1 = more deterministic, >1 = more creative, 1 = raw model distribution
        - top_k: only consider the k most likely tokens (prevents sampling rare garbage)
        
        This is inherently sequential - each token depends on all previous ones.
        That's why inference optimization matters so much for generation.
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids[:, -self.max_seq_len:]  # Crop to max context window
            output = self(idx_cond)
            logits = output['logits'][:, -1, :] / temperature  # Only the last position

            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')  # Zero out everything outside top-k

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # Sample, don't argmax
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 1.3 Data: WikiText-2 + BPE Tokenizer

**WikiText-2** is a collection of high-quality Wikipedia articles (~2M tokens in the training split). It's small enough to download in seconds, clean enough to skip preprocessing, and is the standard benchmark for language model papers - GPT-2 and Transformer-XL both report perplexity on it.

**BPE (Byte Pair Encoding)** tokenizer: starts with individual characters and iteratively merges the most frequent adjacent pairs. Common words like "the" become single tokens; rare words like "unforgettable" get split into subwords like ["un", "forget", "table"]. This means the tokenizer never encounters a truly unknown word.

**Production pattern**: Train the tokenizer on the training split only, then use it consistently across train/val/test. Fitting the tokenizer on test data is a subtle data leak - the vocabulary would reflect test-set word frequencies.

> **Why vocab_size=8192?** Larger vocab = shorter sequences (common phrases are single tokens) but more embedding parameters. 8K is a pragmatic choice for workshop scale. Production models use 32K-256K.

In [ ]:
from src.data import prepare_wikitext2, create_dataloaders

train_dataset, val_dataset, test_dataset, tokenizer = prepare_wikitext2(
    vocab_size=8192,
    seq_len=128,
    tokenizer_path='../tokenizer.json',
)

print(f"Vocabulary size: {tokenizer.get_vocab_size()}")
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")
print(f"Sequence length: {train_dataset.seq_len}")

In [ ]:
# Inspect a sample: target is input shifted by 1 position (next-token prediction).
# Input tokens 0..127, target tokens 1..128. The model learns to predict
# each next token given all previous tokens. This is how every autoregressive
# language model works - GPT, LLaMA, Claude.
x, y = train_dataset[0]
print(f"Input shape:  {x.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFirst 10 input tokens:  {x[:10].tolist()}")
print(f"First 10 target tokens: {y[:10].tolist()}")
print(f"\nDecoded input:  {tokenizer.decode(x[:20].tolist())}")
print(f"Decoded target: {tokenizer.decode(y[:20].tolist())}")

In [ ]:
BATCH_SIZE = 64

train_loader, val_loader = create_dataloaders(
    train_dataset, val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,  # We'll optimize this in Module 2
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

## 1.4 Instantiate the Model

Let's create our model and inspect its size.

**Where the parameters live:**
- Embedding: 8192 x 256 = ~2.1M (shared with output projection via weight tying)
- 4 decoder layers, each ~526K: Q/K/V projections + FFN up/down + LayerNorm
- Total: ~5M trainable parameters (~20 MB in float32)

For reference: GPT-2 Small is 117M params, GPT-3 is 175B. Same architecture, different dimensions.

In [ ]:
VOCAB_SIZE = tokenizer.get_vocab_size()

model = TransformerLM(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    n_heads=4,
    d_ff=512,
    n_layers=4,
    max_seq_len=128,
    dropout=0.1,
).to(device)

print(f"Model parameters: {model.count_parameters():,}")
print(f"Model size (approx): {model.count_parameters() * 4 / 1024 / 1024:.1f} MB (float32)")
print(f"\nArchitecture:\n{model}")

In [ ]:
# Sanity check: forward pass.
# A random model assigns ~uniform probability to all tokens, so initial loss
# should be -log(1/vocab_size) = log(vocab_size) ~ 9.0 for vocab 8192.
# If your initial loss is way off, something is wrong with your data pipeline
# or architecture. Always check this before training.
sample_x, sample_y = next(iter(train_loader))
sample_x, sample_y = sample_x.to(device), sample_y.to(device)

output = model(sample_x, targets=sample_y)
print(f"Logits shape: {output['logits'].shape}")
print(f"Loss: {output['loss'].item():.4f}")
print(f"Expected initial loss (random): {math.log(VOCAB_SIZE):.4f}")

## 1.5 The Training Loop

A production training loop is more than `loss.backward()` followed by `optimizer.step()`. Here's what separates a tutorial from production:

1. **Learning rate schedule** - Linear warmup (first ~5% of steps) prevents early instability when the loss landscape is chaotic, then cosine decay smoothly reduces the LR. This is the standard for transformer training (GPT-3, LLaMA, etc.).

2. **Gradient clipping** (`max_norm=1.0`) - Limits the total gradient norm to prevent one bad batch from destroying all weights. Without it, a single extreme gradient can push weights to NaN in one step. Non-negotiable for transformers.

3. **Periodic evaluation** - Track val_loss and perplexity every epoch. A growing gap between train and val loss = overfitting. Perplexity is `exp(loss)` and represents "how many tokens the model is choosing between" - lower is better.

4. **Structured logging** - Logs go to both console and timestamped files. In a production job that runs overnight, `print()` output is lost. Log files survive crashes.

5. **Checkpointing** - Save model state, optimizer state (needed for resume), epoch, and val_loss. Auto-delete old checkpoints (`keep_last_n=3`) so disk doesn't fill up.

6. **Qualitative samples** - Generate text each epoch. Loss numbers can lie - the model might have low loss but produce repetitive garbage. Seeing actual output catches problems that metrics miss.

In [ ]:
from src.utils import setup_logging, CheckpointManager, MetricsTracker
from src.evaluate import evaluate, generate_sample

# --- Hyperparameters ---
EPOCHS = 3
LEARNING_RATE = 3e-4      # The "safe default" for Adam with transformers
WEIGHT_DECAY = 0.01        # L2 regularization - penalizes large weights
WARMUP_STEPS = 100         # ~5-10% of total steps is typical
MAX_GRAD_NORM = 1.0        # Standard for transformers (GPT-2, LLaMA all use 1.0)
LOG_INTERVAL = 50

# --- Optimizer + Scheduler ---
# AdamW decouples weight decay from the gradient update (Loshchilov & Hutter, 2019).
# Standard Adam applies weight decay inside the gradient, which interacts poorly
# with the adaptive learning rate. AdamW fixes this. Always use AdamW, not Adam.
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS

def cosine_with_warmup(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_with_warmup)

# --- Logging + Checkpointing ---
logger = setup_logging('../logs')
metrics = MetricsTracker('../logs')
ckpt_manager = CheckpointManager('../checkpoints', keep_last_n=3)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {WARMUP_STEPS}")

In [ ]:
# --- Training Loop ---
global_step = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (input_ids, targets) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        # Forward
        output = model(input_ids, targets=targets)
        loss = output['loss']

        # Backward: the order here matters!
        loss.backward()                         # Compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)  # Cap gradient magnitude
        optimizer.step()                        # Update weights
        optimizer.zero_grad(set_to_none=True)   # Deallocates grads (faster than filling with zeros)
        scheduler.step()                        # Advance learning rate schedule

        epoch_loss += loss.item()
        global_step += 1

        if global_step % LOG_INTERVAL == 0:
            avg_loss = epoch_loss / (batch_idx + 1)
            ppl = math.exp(min(avg_loss, 20))
            lr = scheduler.get_last_lr()[0]
            logger.info(
                f"epoch {epoch+1} | step {global_step:>5d} | "
                f"loss {loss.item():.4f} | ppl {ppl:.1f} | lr {lr:.2e}"
            )
            metrics.log({
                'epoch': epoch + 1, 'step': global_step,
                'train_loss': loss.item(), 'ppl': ppl, 'lr': lr,
            })

    # --- End of Epoch Evaluation ---
    val_metrics = evaluate(model, val_loader, device)
    epoch_time = time.time() - epoch_start

    logger.info(
        f"\nEpoch {epoch+1}/{EPOCHS} done in {epoch_time:.1f}s | "
        f"val_loss {val_metrics['val_loss']:.4f} | "
        f"val_ppl {val_metrics['val_perplexity']:.1f}"
    )

    # Checkpoint
    if val_metrics['val_loss'] < best_val_loss:
        best_val_loss = val_metrics['val_loss']
        logger.info(f"  New best val_loss: {best_val_loss:.4f}")

    ckpt_manager.save(
        model, optimizer, epoch + 1, global_step,
        val_metrics['val_loss'],
    )

    # Qualitative check
    sample = generate_sample(model, tokenizer, 'The', device, max_new_tokens=30)
    logger.info(f"  Sample: {sample[:120]}")

logger.info(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")
metrics.save()

## 1.6 Inspect What We Built

A production training run produces more than just a model. Let's look at the artifacts:

- **`logs/metrics.jsonl`** - JSONL format (one JSON object per line) so you can append without loading the whole file. Easy to parse, plot, or feed into dashboards.
- **`checkpoints/checkpoint_*.pt`** - Contains model weights, optimizer state (needed for resume), epoch, step, and val_loss. Only the last 3 are kept (`keep_last_n=3`).
- **`logs/train_*.log`** - Timestamped human-readable logs for debugging after the fact.

In [ ]:
import json
from pathlib import Path

# Training metrics
metrics_file = Path('../logs/metrics.jsonl')
if metrics_file.exists():
    entries = [json.loads(line) for line in metrics_file.read_text().strip().split('\n')]
    print(f"Logged {len(entries)} metric entries")
    print(f"\nLast entry: {entries[-1]}")

# Checkpoints
ckpts = sorted(Path('../checkpoints').glob('checkpoint_*.pt'))
for ckpt in ckpts:
    size_mb = ckpt.stat().st_size / 1024 / 1024
    print(f"  {ckpt.name} ({size_mb:.1f} MB)")

In [ ]:
# Load from checkpoint and generate.
# This is the same pattern you'd use to resume training after a crash,
# or to load a trained model for inference in a completely different script.
checkpoint = ckpt_manager.load(model)
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, val_loss {checkpoint['val_loss']:.4f}")

prompts = ['The president', 'In the year', 'Scientists discovered']
for prompt in prompts:
    text = generate_sample(model, tokenizer, prompt, device, max_new_tokens=40)
    print(f"\n> {prompt}")
    print(f"  {text[:150]}")

## Key Takeaways

What we built vs. a typical tutorial:

| Tutorial | Production |
|----------|------------|
| `print(loss)` | Structured logging to file |
| Constant learning rate | Cosine schedule with warmup |
| No gradient clipping | `clip_grad_norm_` always on |
| `torch.save(model)` | Checkpoint manager with auto-cleanup |
| No eval during training | Periodic eval + perplexity tracking |
| Hardcoded hyperparams | Config files (YAML) |
| `set_seed` missing | Full reproducibility setup |

**Next up**: Module 2 - making this training pipeline 2-5x faster and stable enough to run unattended.